# Train Oddmouth on a Kaggle GPU

Settings: **Accelerator → GPU T4 x2** (or any GPU) and **Internet → On**. Then *Save Version → Save & Run All* to train in the background.

To continue earlier training, attach the checkpoints with **Add Input**: either your uploaded checkpoint dataset or this notebook's previous output. The highest step for each stage is used.

Checkpoints and samples are written to `/kaggle/working/runs` (the notebook's Output tab). Keep the notebook and its output **private**: the default DailyTalk data is not cleared for commercial use.

In [ ]:
# Settings
HOURS = 11          # Kaggle sessions end at 12 h; checkpoints are backed up every 30 minutes when HF_TOKEN is set
BRANCH = 'main'
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Optional backup: add a Hugging Face write token as the Kaggle secret HF_TOKEN (Add-ons -> Secrets)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN found: checkpoints will be backed up to a private Hugging Face repo')
except Exception:
    print('No HF_TOKEN secret: checkpoints are kept only if this version finishes')

In [ ]:
# 1. Install espeak-ng and the two Python packages Kaggle lacks, then fetch the code
!apt-get -qq update && apt-get -qq install -y espeak-ng > /dev/null
!pip -q install phonemizer==3.3.0 soundfile
!rm -rf /tmp/oddmouth && git clone -q --depth 1 -b {BRANCH} https://github.com/farzeenilyaszargar/oddmouth.git /tmp/oddmouth
%cd /tmp/oddmouth
!git log --oneline -1

In [ ]:
# 2. Stream and prepare DailyTalk (about 770 MB of features; nothing is saved to the output)
!python -u scripts/prepare_dailytalk.py --root /tmp/oddmouth-data --cpu-share 1.0 2>&1 | tail -n 5

In [ ]:
# 3. Train both stages until HOURS runs out (saves every 10 minutes, backs up every 30)
!python -u scripts/train_cloud.py --hours {HOURS} --runs /kaggle/working/runs --data /tmp/oddmouth-data --resume-from /kaggle/input

In [ ]:
# 4. Listen to the newest samples
from pathlib import Path
from IPython.display import Audio, display
for stage in ('vocoder', 'acoustic'):
    files = sorted(Path(f'/kaggle/working/runs/{stage}/samples').glob('*.wav'))
    for path in files[-2:]:
        print(stage, path.name)
        display(Audio(str(path)))